# LLM JSON Schema — Structured Output

Extract structured metadata from a real academic paper PDF, constraining each provider's output to the same JSON schema so the response always parses cleanly to a Python dict — no regex, no hallucinated keys.

Each provider has its own mechanism; all share the same schema and prompt.

In [1]:
import json
import os
from pypdf import PdfReader

with open('paper_schema.json') as f:
    SCHEMA = json.load(f)

reader = PdfReader('Towards_a_Platform_for_AI_Assisted_Papyrology.pdf')
pdf_text = '\n'.join(page.extract_text() for page in reader.pages)

SCHEMA_PROMPT = f"""
Extract structured metadata from the following academic paper:

{pdf_text}
"""

---
## OpenAI

Passed as `response_format` with `type='json_schema'`. The SDK validates the response automatically when `strict=True`.

In [ ]:
from openai import OpenAI

OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')
OPENAI_MODEL   = 'gpt-5'

openai_client = OpenAI(api_key=OPENAI_API_KEY)
openai_resp = openai_client.chat.completions.create(
    model=OPENAI_MODEL,
    messages=[
        {
            'role': 'user',
            'content': SCHEMA_PROMPT
        }
    ],
    response_format={
        'type': 'json_schema',
        'json_schema': {
            'name': 'book_recommendation',
            'strict': True,
            'schema': SCHEMA
        }
    }
)
result = json.loads(
    openai_resp.choices[0].message.content
)
print(json.dumps(result, indent=2))

{
  "title": "Towards a Platform for AI-Assisted Papyrology",
  "authors": [
    "Matthew I. Swindall",
    "Graham West",
    "James H. Brusuelas",
    "Alex C. Williams",
    "John F. Wallin"
  ],
  "year": 2024,
  "venue": "Joint Proceedings of the ACM IUI Workshops 2024 (CEUR Workshop Proceedings)",
  "keywords": [
    "Digital Humanities",
    "Machine Learning",
    "Papyrology",
    "Generative AI",
    "Natural Language Processing",
    "Transfer Learning",
    "Handwritten Text Recognition",
    "Blockchain & Smart Contracts"
  ],
  "summary": "The paper presents a vision and initial components of an AI-powered platform to assist papyrologists with transcription, dating, identification, and editing of ancient manuscripts, focusing on Greek papyri. It introduces datasets and an HTR pipeline, demonstrates synthetic data augmentation with GANs to mitigate sampling bias, proposes a fragment-dating pipeline, and outlines future NLP and blockchain-based edition management directions

---
## Anthropic

Uses `output_config` to attach the JSON schema directly to the request. The SDK also offers `client.messages.parse()` for automatic Pydantic model validation.

In [3]:
from anthropic import Anthropic

ANTHROPIC_API_KEY = os.getenv('ANTHROPIC_API_KEY')
ANTHROPIC_MODEL   = 'claude-sonnet-4-5-20250929'

anthropic_client = Anthropic(api_key=ANTHROPIC_API_KEY)
anthropic_resp = anthropic_client.messages.create(
    model=ANTHROPIC_MODEL,
    max_tokens=500,
    messages=[{'role': 'user', 'content': SCHEMA_PROMPT}],
    output_config={'format': {'type': 'json_schema', 'schema': SCHEMA}}
)
result = json.loads(anthropic_resp.content[0].text)
print(json.dumps(result, indent=2))

{
  "title": "Towards a Platform for AI-Assisted Papyrology",
  "authors": [
    "Matthew I. Swindall",
    "Graham West",
    "James H. Brusuelas",
    "Alex C. Williams",
    "John F. Wallin"
  ],
  "year": 2024,
  "venue": "Joint Proceedings of the ACM IUI Workshops 2024",
  "keywords": [
    "Digital Humanities",
    "Machine Learning",
    "Papyrology",
    "Generative AI",
    "Natural Language Processing",
    "Transfer Learning",
    "Handwritten Text Recognition",
    "Blockchain & Smart Contracts"
  ],
  "summary": "This paper proposes an AI-powered platform to assist experts in transcribing, dating, identifying, and editing ancient Greek papyri manuscripts. The authors present their ongoing work on machine learning tools including handwritten text recognition pipelines, automated manuscript dating using ResNet and Gaussian Process models, and a blockchain-based system for managing digital editions. The platform aims to make AI tools accessible to scholars and be extensible t

---
## Google Gemini

Pass the schema to `response_schema` in `GenerateContentConfig` (alongside `response_mime_type='application/json'`). Gemini enforces the structure natively — no need to inject the schema into the prompt. (`additionalProperties` is dropped, since Gemini's schema subset doesn't accept it.)

In [ ]:
from google import genai
from google.genai import types

GEMINI_API_KEY = os.getenv('GEMINI_API_KEY')
GOOGLE_MODEL   = 'gemini-2.5-flash'

client = genai.Client(api_key=GEMINI_API_KEY)

# Gemini's response_schema follows the OpenAPI subset and rejects `additionalProperties`
# (kept in SCHEMA for OpenAI strict mode), so drop that one key.
gemini_schema = {k: v for k, v in SCHEMA.items() if k != 'additionalProperties'}

gemini_resp = client.models.generate_content(
    model=GOOGLE_MODEL,
    contents=SCHEMA_PROMPT,
    config=types.GenerateContentConfig(
        response_mime_type='application/json',
        response_schema=gemini_schema,
    ),
)
result = json.loads(gemini_resp.text)
print(json.dumps(result, indent=2))

{
  "title": "Towards a Platform for AI-Assisted Papyrology",
  "authors": [
    "Matthew I. Swindall",
    "Graham West",
    "James H. Brusuelas",
    "Alex C. Williams",
    "John F. Wallin"
  ],
  "year": 2024,
  "venue": "Joint Proceedings of the ACM IUI Workshops 2024, Greenville, South Carolina, USA",
  "keywords": [
    "Digital Humanities",
    "Machine Learning",
    "Papyrology",
    "Generative AI",
    "Natural Language Processing",
    "Transfer Learning",
    "Handwritten Text Recognition",
    "Blockchain & Smart Contracts"
  ],
  "summary": "This paper proposes an AI-powered platform designed to assist experts in transcribing, dating, identifying, and editing ancient manuscripts, with a focus on Greek papyrology. The authors discuss their ongoing work and envision a broader, intuitive application that can be extended to additional languages and media. The platform aims to be an all-in-one system for AI-assisted papyrology.",
  "methods": [
    "Deep learning",
    "Han

---
## Ollama (local)

Pass the JSON schema directly to the `format=` parameter. Ollama constrains the tokenizer at the grammar level — the output is always valid JSON.

In [ ]:
import ollama

OLLAMA_MODEL = 'llama3.2:3b-instruct-q5_K_M'

ollama_resp = ollama.chat(
    model=OLLAMA_MODEL,
    messages=[{'role': 'user', 'content': SCHEMA_PROMPT}],
    format=SCHEMA,
    options={'temperature': 0.2}
)
result = json.loads(ollama_resp['message']['content'])
print(json.dumps(result, indent=2))

{
  "title": "Towards a Platform for AI-Assisted Papyrology",
  "authors": [
    "Matthew I. Swindall",
    "Graham West",
    "James H. Brusuelas",
    "Alex C. Williams",
    "John F. Wallin"
  ],
  "year": 2024,
  "venue": "Joint Proceedings of the ACM IUI Workshops 2024, March 18-21, 2024, Greenville, South Carolina, USA",
  "keywords": [
    "Digital Humanities",
    "Machine Learning",
    "Papyrology",
    "Generative AI",
    "Natural Language Processing",
    "Transfer Learning",
    "Handwritten Text Recognition",
    "Blockchain & Smart Contracts"
  ],
  "summary": "The paper proposes an AI-powered platform to assist experts in transcribing, dating, identifying, and editing ancient manuscripts. It discusses ongoing work on AI-assisted Greek papyrology and a vision for broader application.",
  "methods": [
    "Deep Learning",
    "Transfer Learning",
    "Generative Adversarial Networks (GANs)",
    "Handwritten Text Recognition (HTR)",
    "Blockchain & Smart Contracts"
  ]